In [ ]:
import arcpy
import os
import re
import datetime
import calendar
from datetime import datetime

In [ ]:
#County
project = arcpy.mp.ArcGISProject("CURRENT")
gdb = project.defaultGeodatabase
project_folder = os.path.dirname(project.filePath)

county_statistics = os.path.join(gdb, "CountyStatistics") #Should already be created


# ============================================================
# INPUT / OUTPUT
# ============================================================

# Folder containing the already-created zonal statistics tables
zonal_folder = os.path.join(
    project_folder,
    "ZonalStatsTables"
)

# ============================================================
# CHECK INPUTS
# ============================================================

if not os.path.exists(zonal_folder):
    raise FileNotFoundError(
        f"ZonalStatsTables folder not found:\n{zonal_folder}"
    )

if not arcpy.Exists(county_statistics):
    raise FileNotFoundError(
        f"CountyStatistics table not found:\n{county_statistics}"
    )


print()
print("============================================================")
print("IMPORTING ZONAL STATISTICS")
print("============================================================")
print()
print(f"Source folder:")
print(zonal_folder)
print()
print(f"Destination:")
print(county_statistics)
print()


# ============================================================
# MAKE SURE COUNTY STATISTICS HAS REQUIRED FIELDS
# ============================================================

required_fields = {
    "GEOID": "TEXT",
    "START": "DATE",
    "END": "DATE",
    "PERIOD": "TEXT",
    "VARIABLE": "TEXT",
    "VALUE": "DOUBLE"
}

existing_fields = {
    field.name.upper()
    for field in arcpy.ListFields(county_statistics)
}

for field_name, field_type in required_fields.items():

    if field_name.upper() not in existing_fields:

        print(f"Adding missing field: {field_name}")

        arcpy.management.AddField(
            county_statistics,
            field_name,
            field_type
        )


# ============================================================
# FIND ALL ZONAL STATISTICS TABLES
# ============================================================

tables = sorted(
    file
    for file in os.listdir(zonal_folder)
    if file.lower().endswith(".dbf")
)

print(f"Found {len(tables)} zonal statistics tables.")
print()


# ============================================================
# PROCESS EACH TABLE
# ============================================================

for table_file in tables:

    table_path = os.path.join(
        zonal_folder,
        table_file
    )

    table_name = os.path.splitext(
        table_file
    )[0]

    print("------------------------------------------------------------")
    print(f"Table: {table_name}")


    # ========================================================
    # EXTRACT DATE
    #
    # Example:
    #
    # Zonal_tmin_prism_tmin_us_30s_202608
    #
    #                       YYYYMM
    # ========================================================

    date_match = re.search(
        r"(\d{4})(\d{2})$",
        table_name
    )

    if not date_match:

        print("  SKIPPED: Could not find YYYYMM in table name.")
        continue


    year = int(date_match.group(1))
    month = int(date_match.group(2))


    # Validate month
    if month < 1 or month > 12:

        print(
            f"  SKIPPED: Invalid month: {month}"
        )
        continue


    # ========================================================
    # EXTRACT VARIABLE
    #
    # Zonal_tmin_prism_tmin_us_30s_202608
    #       ↑
    #     tmin
    # ========================================================

    parts = table_name.split("_")

    if len(parts) < 3:

        print("  SKIPPED: Unexpected table name.")
        continue


    variable = parts[1]


    # ========================================================
    # CREATE START / END DATES
    # ========================================================

    start_date = datetime(
        year,
        month,
        1
    )

    last_day = calendar.monthrange(
        year,
        month
    )[1]

    end_date = datetime(
        year,
        month,
        last_day
    )

    period = "historical monthly"


    print(f"  Variable: {variable}")
    print(f"  Start:    {start_date.date()}")
    print(f"  End:      {end_date.date()}")
    print(f"  Period:   {period}")


    # ========================================================
    # FIND GEOID + MEAN FIELDS
    # ========================================================

    table_fields = {
        field.name.upper(): field.name
        for field in arcpy.ListFields(table_path)
    }


    if "GEOID" not in table_fields:

        print("  SKIPPED: GEOID field not found.")
        continue


    if "MEAN" not in table_fields:

        print("  SKIPPED: MEAN field not found.")
        continue


    geoid_field = table_fields["GEOID"]
    mean_field = table_fields["MEAN"]


    # ========================================================
    # APPEND RECORDS
    # ========================================================

    inserted = 0


    with arcpy.da.SearchCursor(
        table_path,
        [geoid_field, mean_field]
    ) as search_cursor:

        with arcpy.da.InsertCursor(
            county_statistics,
            [
                "GEOID",
                "START",
                "END",
                "PERIOD",
                "VARIABLE",
                "VALUE"
            ]
        ) as insert_cursor:

            for geoid, mean_value in search_cursor:

                insert_cursor.insertRow(
                    (
                        str(geoid),
                        start_date,
                        end_date,
                        period,
                        variable,
                        mean_value
                    )
                )

                inserted += 1


    print(f"  Added {inserted} records.")


# ============================================================
# COMPLETE
# ============================================================

print()
print("============================================================")
print("COMPLETE")
print("============================================================")
print()
print(f"Processed folder:")
print(zonal_folder)
print()
print(f"Updated table:")
print(county_statistics)
print()

In [ ]:
#Fields
#Need to add County_ID field
# ============================================================
# PROJECT SETUP
# ============================================================

project = arcpy.mp.ArcGISProject("CURRENT")
gdb = project.defaultGeodatabase
project_folder = os.path.dirname(project.filePath)

field_statistics = os.path.join(
    gdb,
    "FieldStatistics"
)


# ============================================================
# INPUT / OUTPUT
# ============================================================

# Folder containing the already-created field zonal
# statistics tables
zonal_folder = os.path.join(
    project_folder,
    "ZonalStatsTablesFieldsNew"
)


# ============================================================
# CHECK INPUTS
# ============================================================

if not os.path.exists(zonal_folder):
    raise FileNotFoundError(
        f"ZonalStatsTables folder not found:\n{zonal_folder}"
    )

if not arcpy.Exists(field_statistics):
    raise FileNotFoundError(
        f"FieldStatistics table not found:\n{field_statistics}"
    )


print()
print("============================================================")
print("IMPORTING FIELD ZONAL STATISTICS")
print("============================================================")
print()
print("Source folder:")
print(zonal_folder)
print()
print("Destination:")
print(field_statistics)
print()


# ============================================================
# MAKE SURE FIELD STATISTICS HAS REQUIRED FIELDS
# ============================================================

required_fields = {
    "FIELD_ID": "TEXT",
    "START": "DATE",
    "END": "DATE",
    "PERIOD": "TEXT",
    "VARIABLE": "TEXT",
    "VALUE": "DOUBLE"
}


existing_fields = {
    field.name.upper()
    for field in arcpy.ListFields(field_statistics)
}


for field_name, field_type in required_fields.items():

    if field_name.upper() not in existing_fields:

        print(f"Adding missing field: {field_name}")

        arcpy.management.AddField(
            field_statistics,
            field_name,
            field_type
        )


# ============================================================
# FIND ALL ZONAL STATISTICS TABLES
# ============================================================

tables = sorted(
    file
    for file in os.listdir(zonal_folder)
    if file.lower().endswith(".dbf")
)


print(
    f"Found {len(tables)} zonal statistics tables."
)
print()


# ============================================================
# PROCESS EACH TABLE
# ============================================================

for table_file in tables:

    table_path = os.path.join(
        zonal_folder,
        table_file
    )

    table_name = os.path.splitext(
        table_file
    )[0]


    print(
        "------------------------------------------------------------"
    )

    print(
        f"Table: {table_name}"
    )


    # ========================================================
    # EXTRACT DATE
    #
    # Example:
    #
    # Zonal_tmin_prism_tmin_us_30s_202608
    #
    #                       YYYYMM
    # ========================================================

    date_match = re.search(
        r"(\d{4})(\d{2})$",
        table_name
    )


    if not date_match:

        print(
            "  SKIPPED: Could not find YYYYMM in table name."
        )

        continue


    year = int(
        date_match.group(1)
    )

    month = int(
        date_match.group(2)
    )


    # ========================================================
    # VALIDATE MONTH
    # ========================================================

    if month < 1 or month > 12:

        print(
            f"  SKIPPED: Invalid month: {month}"
        )

        continue


    # ========================================================
    # EXTRACT VARIABLE
    #
    # Zonal_tmin_prism_tmin_us_30s_202608
    #       ↑
    #     tmin
    # ========================================================

    parts = table_name.split("_")


    if len(parts) < 3:

        print(
            "  SKIPPED: Unexpected table name."
        )

        continue


    variable = parts[1]


    # ========================================================
    # CREATE START / END DATES
    # ========================================================

    start_date = datetime(
        year,
        month,
        1
    )


    last_day = calendar.monthrange(
        year,
        month
    )[1]


    end_date = datetime(
        year,
        month,
        last_day
    )


    period = "monthly"


    print(
        f"  Variable: {variable}"
    )

    print(
        f"  Start:    {start_date.date()}"
    )

    print(
        f"  End:      {end_date.date()}"
    )

    print(
        f"  Period:   {period}"
    )


    # ========================================================
    # FIND FIELD_ID + MEAN FIELDS
    # ========================================================

    table_fields = {
        field.name.upper(): field.name
        for field in arcpy.ListFields(table_path)
    }


    if "FIELD_ID" not in table_fields:

        print(
            "  SKIPPED: FIELD_ID field not found."
        )

        continue


    if "MEAN" not in table_fields:

        print(
            "  SKIPPED: MEAN field not found."
        )

        continue


    field_id_field = table_fields["FIELD_ID"]
    mean_field = table_fields["MEAN"]


    # ========================================================
    # APPEND RECORDS
    # ========================================================

    inserted = 0


    with arcpy.da.SearchCursor(
        table_path,
        [
            field_id_field,
            mean_field
        ]
    ) as search_cursor:

        with arcpy.da.InsertCursor(
            field_statistics,
            [
                "FIELD_ID",
                "START",
                "END",
                "PERIOD",
                "VARIABLE",
                "VALUE"
            ]
        ) as insert_cursor:

            for field_id, mean_value in search_cursor:

                insert_cursor.insertRow(
                    (
                        str(field_id),
                        start_date,
                        end_date,
                        period,
                        variable,
                        mean_value
                    )
                )

                inserted += 1


    print(
        f"  Added {inserted} records."
    )


# ============================================================
# COMPLETE
# ============================================================

print()
print("============================================================")
print("COMPLETE")
print("============================================================")
print()

print(
    "Processed folder:"
)

print(
    zonal_folder
)

print()

print(
    "Updated table:"
)

print(
    field_statistics
)

print()